In [2]:
import os
import pickle
import pandas as pd
import numpy as np
from collections import Counter
from tqdm.auto import tqdm
import ast

# ================= 配置路径 =================
BASE_DIR = "/home/tyj/project/code_TangYuJie/processed_data/CIC-IDS-2017-session-srcIP_srcPort_dstIP_dstPort_proto"
INFO_PATH = os.path.join(BASE_DIR, "all_session_graph_info.pkl")
SESSION_SPLIT_CSV = os.path.join(BASE_DIR, "all_split_session.csv")
FLOW_CSV = os.path.join(BASE_DIR, "all_flow.csv")

def analyze_distribution():
    print(f"🔍 正在读取数据集信息: {BASE_DIR}")

    # 1. 加载图（Session）信息
    with open(INFO_PATH, 'rb') as f:
        info_data = pickle.load(f)

    label_names = info_data.get('label_name', [])
    label_ids = info_data.get('label_id', [])
    train_idx = info_data.get('train_index', [])
    val_idx = info_data.get('validate_index', [])
    test_idx = info_data.get('test_index', [])

    # 映射 ID 到名称
    unique_ids = sorted(list(set(label_ids)))
    id_to_name = {}
    for lid, name in zip(label_ids, label_names):
        if lid not in id_to_name:
            id_to_name[lid] = name

    # 统计图分布
    def count_graphs(indices):
        c = Counter([label_ids[i] for i in indices])
        return [c.get(lid, 0) for lid in unique_ids]

    graph_counts = {
        'train': count_graphs(train_idx),
        'val': count_graphs(val_idx),
        'test': count_graphs(test_idx)
    }

    # 2. 解析 Session 划分与 Flow UID 的对应关系
    print("⏳ 正在建立 Flow UID 到划分的映射 (Split Mapping)...")
    split_df = pd.read_csv(SESSION_SPLIT_CSV, usecols=['flow_uid_list', 'split'])
    uid_to_split = {}

    for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc="解析 Session CSV"):
        split = row['split']
        uids = row['flow_uid_list']
        if isinstance(uids, str):
            uids = ast.literal_eval(uids)
        for uid in uids:
            uid_to_split[uid] = split

    # 3. 统计流（Flow）标签分布 (先用 raw label 统计)
    print("⏳ 正在读取 Flow CSV 统计流样本数量...")
    raw_flow_stats = {'train': Counter(), 'val': Counter(), 'test': Counter()}

    chunk_size = 500000
    for chunk in tqdm(pd.read_csv(FLOW_CSV, usecols=['uid', 'label'], chunksize=chunk_size), desc="扫描 Flow CSV"):
        chunk['label'] = chunk['label'].astype(str)
        # 批量处理该块
        for _, row in chunk.iterrows():
            split = uid_to_split.get(row['uid'])
            if split:
                s_key = 'val' if split == 'validate' else split
                raw_flow_stats[s_key][row['label']] += 1

    # ================= 核心修复：标签映射归类 =================
    flow_stats = {'train': Counter(), 'val': Counter(), 'test': Counter()}
    unmapped_labels = set()

    for s_key in ['train', 'val', 'test']:
        for raw_label, count in raw_flow_stats[s_key].items():
            mapped_name = None
            # 转为小写并替换部分符号，便于比较
            raw_norm = str(raw_label).lower().replace('_', ' ').replace('-', ' ')

            # 尝试与 id_to_name 中的标准名称进行匹配
            for lid, name in id_to_name.items():
                name_norm = name.lower().replace('_', ' ').replace('-', ' ')

                # 如果名称完全一致，或者图的名称包含在流的原始名称中 (例如 "web attack" 在 "web attack \xbf brute force" 中)
                if name_norm == raw_norm or name_norm in raw_norm:
                    mapped_name = name
                    break

            # 备选的无空格暴力匹配
            if not mapped_name:
                for lid, name in id_to_name.items():
                    if name.lower().replace(' ', '') in raw_norm.replace(' ', ''):
                        mapped_name = name
                        break

            if mapped_name:
                flow_stats[s_key][mapped_name] += count
            else:
                unmapped_labels.add(raw_label)
                flow_stats[s_key][raw_label] += count # 如果真的匹配不上，保留原名

    if unmapped_labels:
        print(f"\n⚠️ 警告: 有以下原始标签未能映射到你图(PKL)中的名称类别里: {unmapped_labels}")

    # 4. 汇总结果并打印表格
    print("\n" + "="*105)
    print(f"{'类别名称 (ID)':<25} | {'图(Train)':<10} | {'图(Val)':<10} | {'图(Test)':<10} || {'流(Train)':<10} | {'流(Val)':<10} | {'流(Test)':<10}")
    print("-" * 105)

    for i, lid in enumerate(unique_ids):
        name = id_to_name[lid]
        # 图数量
        g_t, g_v, g_te = graph_counts['train'][i], graph_counts['val'][i], graph_counts['test'][i]
        # 流数量 (现在由于映射，可以正确匹配上 name 了)
        f_t = flow_stats['train'].get(name, 0)
        f_v = flow_stats['val'].get(name, 0)
        f_te = flow_stats['test'].get(name, 0)

        print(f"{f'{name} ({lid})':<25} | {g_t:<10} | {g_v:<10} | {g_te:<10} || {f_t:<10} | {f_v:<10} | {f_te:<10}")

    # 计算总计
    print("-" * 105)
    sum_gt, sum_gv, sum_gte = len(train_idx), len(val_idx), len(test_idx)
    sum_ft = sum(flow_stats['train'].values())
    sum_fv = sum(flow_stats['val'].values())
    sum_fte = sum(flow_stats['test'].values())
    print(f"{'总计 (Total)':<25} | {sum_gt:<10} | {sum_gv:<10} | {sum_gte:<10} || {sum_ft:<10} | {sum_fv:<10} | {sum_fte:<10}")
    print("="*105)

if __name__ == "__main__":
    analyze_distribution()

🔍 正在读取数据集信息: /home/tyj/project/code_TangYuJie/processed_data/CIC-IDS-2017-session-srcIP_srcPort_dstIP_dstPort_proto
⏳ 正在建立 Flow UID 到划分的映射 (Split Mapping)...


解析 Session CSV:   0%|          | 0/636607 [00:00<?, ?it/s]

⏳ 正在读取 Flow CSV 统计流样本数量...


扫描 Flow CSV: 0it [00:00, ?it/s]


类别名称 (ID)                 | 图(Train)   | 图(Val)     | 图(Test)    || 流(Train)   | 流(Val)     | 流(Test)   
---------------------------------------------------------------------------------------------------------
benign (0)                | 425454     | 596        | 27688      || 437425     | 623        | 28449     
dos hulk (1)              | 162883     | 0          | 0          || 162883     | 0          | 0         
dos slowhttptest (2)      | 16045      | 0          | 0          || 16045      | 0          | 0         
dos slowloris (3)         | 3877       | 0          | 0          || 3877       | 0          | 0         
dos goldeneye (4)         | 7607       | 0          | 0          || 7607       | 0          | 0         
portscan (5)              | 31607      | 63360      | 64015      || 31607      | 63360      | 64015     
ddos (6)                  | 217        | 114        | 86135      || 217        | 114        | 86135     
ftp-patator (7)           | 3986       | 0          |

In [3]:
import dgl
import pandas as pd
import numpy as np
import os

# 1. 设置您的图数据路径
# 请根据您的实际生成路径替换，例如 'processed_data/cicids2017_graphs.bin'
graph_data_path = '/home/tyj/project/code_TangYuJie/processed_data/CIC-IDS-2017-session-srcIP_srcPort_dstIP_dstPort_proto/all_session_graph.bin'

print(f"Loading graphs from {graph_data_path}...")
try:
    # 加载 DGL 图数据
    graphs, label_dict = dgl.load_graphs(graph_data_path)

    # 2. 提取每个图的节点数和边数
    node_counts = [g.num_nodes() for g in graphs]
    edge_counts = [g.num_edges() for g in graphs]

    # 3. 构建 DataFrame
    df_stats = pd.DataFrame({
        'Node_Count': node_counts,
        'Edge_Count': edge_counts
    })

    # 4. 导出为 CSV 文件
    csv_filename = 'cicids2017_graph_stats.csv'
    df_stats.to_csv(csv_filename, index=False)

    print(f"Successfully processed {len(graphs)} graphs.")
    print(f"Data saved to: {os.path.abspath(csv_filename)}")

    # 打印前几行预览
    print("\nData Preview:")
    print(df_stats.head())

except Exception as e:
    print(f"Error loading graph data: {e}")
    print("请确保图文件路径正确且格式为 DGL 支持的 bin/pt 格式。")

Loading graphs from /home/tyj/project/code_TangYuJie/processed_data/CIC-IDS-2017-session-srcIP_srcPort_dstIP_dstPort_proto/all_session_graph.bin...
Successfully processed 899804 graphs.
Data saved to: /home/tyj/project/code_TangYuJie/src/analysis/cicids2017_graph_stats.csv

Data Preview:
   Node_Count  Edge_Count
0           1           1
1           1           1
2           1           1
3           1           1
4           1           1


In [ ]:
import os
import pickle
import pandas as pd
import numpy as np
from collections import Counter
from tqdm.auto import tqdm
import ast

# ================= 配置路径 =================
BASE_DIR = "/home/disk_10T/tyj_data/processed_data/CIC-IoMT-2024-session-srcIP_srcPort_dstIP_dstPort_proto"
INFO_PATH = os.path.join(BASE_DIR, "all_session_graph_info.pkl")
SESSION_SPLIT_CSV = os.path.join(BASE_DIR, "all_split_session.csv")
FLOW_CSV = os.path.join(BASE_DIR, "all_flow.csv")

def analyze_distribution():
    print(f"🔍 正在读取数据集信息: {BASE_DIR}")

    # 1. 加载图（Session）信息
    with open(INFO_PATH, 'rb') as f:
        info_data = pickle.load(f)

    label_names = info_data.get('label_name', [])
    label_ids = info_data.get('label_id', [])
    train_idx = info_data.get('train_index', [])
    val_idx = info_data.get('validate_index', [])
    test_idx = info_data.get('test_index', [])

    # 映射 ID 到名称
    unique_ids = sorted(list(set(label_ids)))
    id_to_name = {}
    for lid, name in zip(label_ids, label_names):
        if lid not in id_to_name:
            id_to_name[lid] = name

    # 统计图分布
    def count_graphs(indices):
        c = Counter([label_ids[i] for i in indices])
        return [c.get(lid, 0) for lid in unique_ids]

    graph_counts = {
        'train': count_graphs(train_idx),
        'val': count_graphs(val_idx),
        'test': count_graphs(test_idx)
    }

    # 2. 解析 Session 划分与 Flow UID 的对应关系
    print("⏳ 正在建立 Flow UID 到划分的映射 (Split Mapping)...")
    split_df = pd.read_csv(SESSION_SPLIT_CSV, usecols=['flow_uid_list', 'split'])
    uid_to_split = {}

    for _, row in tqdm(split_df.iterrows(), total=len(split_df), desc="解析 Session CSV"):
        split = row['split']
        uids = row['flow_uid_list']
        if isinstance(uids, str):
            uids = ast.literal_eval(uids)
        for uid in uids:
            uid_to_split[uid] = split

    # 3. 统计流（Flow）标签分布 (先用 raw label 统计)
    print("⏳ 正在读取 Flow CSV 统计流样本数量...")
    raw_flow_stats = {'train': Counter(), 'val': Counter(), 'test': Counter()}

    chunk_size = 500000
    for chunk in tqdm(pd.read_csv(FLOW_CSV, usecols=['uid', 'label'], chunksize=chunk_size), desc="扫描 Flow CSV"):
        chunk['label'] = chunk['label'].astype(str)
        # 批量处理该块
        for _, row in chunk.iterrows():
            split = uid_to_split.get(row['uid'])
            if split:
                s_key = 'val' if split == 'validate' else split
                raw_flow_stats[s_key][row['label']] += 1

    # ================= 核心修复：标签映射归类 =================
    flow_stats = {'train': Counter(), 'val': Counter(), 'test': Counter()}
    unmapped_labels = set()

    for s_key in ['train', 'val', 'test']:
        for raw_label, count in raw_flow_stats[s_key].items():
            mapped_name = None
            # 转为小写并替换部分符号，便于比较
            raw_norm = str(raw_label).lower().replace('_', ' ').replace('-', ' ')

            # 尝试与 id_to_name 中的标准名称进行匹配
            for lid, name in id_to_name.items():
                name_norm = name.lower().replace('_', ' ').replace('-', ' ')

                # 如果名称完全一致，或者图的名称包含在流的原始名称中 (例如 "web attack" 在 "web attack \xbf brute force" 中)
                if name_norm == raw_norm or name_norm in raw_norm:
                    mapped_name = name
                    break

            # 备选的无空格暴力匹配
            if not mapped_name:
                for lid, name in id_to_name.items():
                    if name.lower().replace(' ', '') in raw_norm.replace(' ', ''):
                        mapped_name = name
                        break

            if mapped_name:
                flow_stats[s_key][mapped_name] += count
            else:
                unmapped_labels.add(raw_label)
                flow_stats[s_key][raw_label] += count # 如果真的匹配不上，保留原名

    if unmapped_labels:
        print(f"\n⚠️ 警告: 有以下原始标签未能映射到你图(PKL)中的名称类别里: {unmapped_labels}")

    # 4. 汇总结果并打印表格
    print("\n" + "="*105)
    print(f"{'类别名称 (ID)':<25} | {'图(Train)':<10} | {'图(Val)':<10} | {'图(Test)':<10} || {'流(Train)':<10} | {'流(Val)':<10} | {'流(Test)':<10}")
    print("-" * 105)

    for i, lid in enumerate(unique_ids):
        name = id_to_name[lid]
        # 图数量
        g_t, g_v, g_te = graph_counts['train'][i], graph_counts['val'][i], graph_counts['test'][i]
        # 流数量 (现在由于映射，可以正确匹配上 name 了)
        f_t = flow_stats['train'].get(name, 0)
        f_v = flow_stats['val'].get(name, 0)
        f_te = flow_stats['test'].get(name, 0)

        print(f"{f'{name} ({lid})':<25} | {g_t:<10} | {g_v:<10} | {g_te:<10} || {f_t:<10} | {f_v:<10} | {f_te:<10}")

    # 计算总计
    print("-" * 105)
    sum_gt, sum_gv, sum_gte = len(train_idx), len(val_idx), len(test_idx)
    sum_ft = sum(flow_stats['train'].values())
    sum_fv = sum(flow_stats['val'].values())
    sum_fte = sum(flow_stats['test'].values())
    print(f"{'总计 (Total)':<25} | {sum_gt:<10} | {sum_gv:<10} | {sum_gte:<10} || {sum_ft:<10} | {sum_fv:<10} | {sum_fte:<10}")
    print("="*105)

if __name__ == "__main__":
    analyze_distribution()

In [ ]:
import dgl
import pandas as pd
import numpy as np
import os

# 1. 设置您的图数据路径
# 请根据您的实际生成路径替换，例如 'processed_data/cicids2017_graphs.bin'
graph_data_path = '/home/disk_10T/tyj_data/processed_data/CIC-IoMT-2024-session-srcIP_srcPort_dstIP_dstPort_proto/all_session_graph.bin'

print(f"Loading graphs from {graph_data_path}...")
try:
    # 加载 DGL 图数据
    graphs, label_dict = dgl.load_graphs(graph_data_path)

    # 2. 提取每个图的节点数和边数
    node_counts = [g.num_nodes() for g in graphs]
    edge_counts = [g.num_edges() for g in graphs]

    # 3. 构建 DataFrame
    df_stats = pd.DataFrame({
        'Node_Count': node_counts,
        'Edge_Count': edge_counts
    })

    # 4. 导出为 CSV 文件
    csv_filename = 'ciciomt2024_graph_stats.csv'
    df_stats.to_csv(csv_filename, index=False)

    print(f"Successfully processed {len(graphs)} graphs.")
    print(f"Data saved to: {os.path.abspath(csv_filename)}")

    # 打印前几行预览
    print("\nData Preview:")
    print(df_stats.head())

except Exception as e:
    print(f"Error loading graph data: {e}")
    print("请确保图文件路径正确且格式为 DGL 支持的 bin/pt 格式。")

In [1]:
import torch;
print(f"MKL可用: {torch.backends.mkl.is_available()}");
print(f"OpenMP可用: {torch.backends.openmp.is_available()}")

MKL可用: True
OpenMP可用: True


In [1]:
import dgl
import torch
import pprint
from dgl.data.utils import load_graphs, load_info

# 1. 定义你的生成文件路径
bin_path = "/home/tyj/project/code_TangYuJie/processed_data/CIC-IDS-2017-session-srcIP_srcPort_dstIP_dstPort_proto/all_session_graph.bin"
info_path = "/home/tyj/project/code_TangYuJie/processed_data/CIC-IDS-2017-session-srcIP_srcPort_dstIP_dstPort_proto/all_session_graph_info.pkl"

# 2. 加载数据
print("正在加载图数据，这可能需要几十秒...")
graphs, _ = load_graphs(bin_path)
info = load_info(info_path)

print(f"✅ 成功加载了 {len(graphs)} 张图。")
print("=" * 60)

# 3. 抽取第一张图进行宏观分析
g = graphs[0]
print(f"【图 0 宏观信息】")
print(f" - 节点数 (Flows): {g.num_nodes()}")
print(f" - 边数 (Relations): {g.num_edges()}")
print(f" - 标签 (Label): {info['label_name'][0]} (ID: {info['label_id'][0]})")
print(f" - 数据划分 (Split): {info['split'][0]}")
print(f" - 包含的节点特征键 (g.ndata.keys): {list(g.ndata.keys())}")
print("=" * 60)

# 4. 详细打印核心特征的张量结构 (查看第 0 个节点)
node_idx = 0
print(f"【节点 {node_idx} 的底层特征异构性分析】\n")

# 我们挑选最典型的三个异构特征来观察
features_to_check = ['flow_numeric_features', 'flow_categorical_features', 'packet_len_seq']

for feat_name in features_to_check:
    if feat_name in g.ndata:
        feat_tensor = g.ndata[feat_name]
        print(f"🔹 特征名称: {feat_name}")
        print(f"   - 全局张量形状 (Shape): {feat_tensor.shape}  <-- 注意这里的维度")
        print(f"   - 数据类型 (Dtype): {feat_tensor.dtype}")

        # 提取第 0 个节点的前 10 个元素展示
        sample_values = feat_tensor[node_idx][:10].tolist()

        # 格式化打印
        if feat_tensor.dtype in [torch.int64, torch.int32, torch.long]:
            print(f"   - 节点 {node_idx} 取值 (前10个): {[int(x) for x in sample_values]} ... (类别索引)")
        else:
            print(f"   - 节点 {node_idx} 取值 (前10个): {[round(float(x), 4) for x in sample_values]} ... (连续数值)")
        print("-" * 40)
    else:
        print(f"🔹 特征名称: {feat_name} (该图中未包含此特征)")
        print("-" * 40)

正在加载图数据，这可能需要几十秒...
✅ 成功加载了 899804 张图。
【图 0 宏观信息】
 - 节点数 (Flows): 1
 - 边数 (Relations): 1
 - 标签 (Label): benign (ID: 0)
 - 数据划分 (Split): train
 - 包含的节点特征键 (g.ndata.keys): ['id', 'flow_numeric_features', 'ssl_textual_attention_mask', 'dns_textual_attention_mask', 'burst_id', 'x509_textual_input_ids', 'domain_probs', 'ssl_textual_input_ids', 'dns_textual_input_ids', 'dns_numeric_features', 'packet_len_seq', 'ts', 'x509_textual_attention_mask', 'packet_iat_seq', 'packet_seq_mask']
【节点 0 的底层特征异构性分析】

🔹 特征名称: flow_numeric_features
   - 全局张量形状 (Shape): torch.Size([1, 88])  <-- 注意这里的维度
   - 数据类型 (Dtype): torch.float32
   - 节点 0 取值 (前10个): [-0.339, -0.0288, -0.0057, -0.0065, -0.0077, -0.0155, -0.0046, -0.0053, -0.339, -0.0077] ... (连续数值)
----------------------------------------
🔹 特征名称: flow_categorical_features (该图中未包含此特征)
----------------------------------------
🔹 特征名称: packet_len_seq
   - 全局张量形状 (Shape): torch.Size([1, 512])  <-- 注意这里的维度
   - 数据类型 (Dtype): torch.float32
   - 节点 0 取值 (前10个): [